<a href="https://colab.research.google.com/github/osergioribeirof/Python/blob/main/SR_GammaFlip__Interativo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
### Rodar essa célula somente uma vez ###
# Delete a # na linha abaixo, execute e coloque de volta a #

#!pip install plotly

In [ ]:
import pandas as pd
import plotly
pd.set_option('plotting.backend','plotly')
import plotly.graph_objs as go
import numpy as np
import scipy
from scipy.stats import norm
#import matplotlib.pyplot as plt
import calendar
from datetime import datetime, timedelta, date

In [ ]:
pd.options.display.float_format = '{:,.4f}'.format

In [ ]:
# Parametros de entrada
filename = 'spx_quotedata.csv'

# Black-Scholes European-Options Gamma
def calcGammaEx(S, K, vol, T, r, q, optType, OI):
    if T == 0 or vol == 0:
        return 0

    dp = (np.log(S/K) + (r - q + 0.5*vol**2)*T) / (vol*np.sqrt(T))
    dm = dp - vol*np.sqrt(T)

    if optType == 'call':
        gamma = np.exp(-q*T) * norm.pdf(dp) / (S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma
    else: # Gamma is same for calls and puts. This is just to cross-check
        gamma = K * np.exp(-r*T) * norm.pdf(dm) / (S * S * vol * np.sqrt(T))
        return OI * 100 * S * S * 0.01 * gamma

def isThirdFriday(d):
    return d.weekday() == 4 and 15 <= d.day <= 21

In [ ]:
# Isso assume que o formato do arquivo CBOE não foi editado, ou seja, a tabela começa na linha 4
optionsFile = open(filename)
optionsFileData = optionsFile.readlines()
optionsFile.close()

In [ ]:
# Extraindo SPX spot
spotLine = optionsFileData[1]
spotPrice = float(spotLine.split('Last:')[1].split(',')[0])
fromStrike = 0.8 * spotPrice
toStrike = 1.2 * spotPrice

In [ ]:
# Extraindo a data de hoje
dateLine = optionsFileData[2]
todayDate = dateLine.split('Date: ')[1].split(',')
monthDay = todayDate[0].split(' ')

In [ ]:
if len(monthDay) == 2:
    year = int(monthDay[4])
    month = monthDay[2]
    day = int(monthDay[0])
else:
    if monthDay[2].isdigit():
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])
    else:
        year = int(monthDay[4])
        month = monthDay[2]
        day = int(monthDay[0])


# criar um dicionário para mapear os nomes dos meses em português para os equivalentes em inglês
nomes_meses = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extrair o nome do mês da string de entrada
nome_mes_pt = monthDay[2]

# converter o nome do mês para inglês usando o dicionário
nome_mes_en = nomes_meses[nome_mes_pt]

# converter o nome do mês para o número correspondente (por exemplo, 'March' -> 3)
num_mes = datetime.strptime(nome_mes_en, '%B').month

# criar o objeto datetime
todayDate = datetime(year=year, month=num_mes, day=day)

In [ ]:
# create a dictionary to map Portuguese month names to English month names
month_names = {'janeiro': 'January', 'fevereiro': 'February', 'março': 'March',
               'abril': 'April', 'maio': 'May', 'junho': 'June',
               'julho': 'July', 'agosto': 'August', 'setembro': 'September',
               'outubro': 'October', 'novembro': 'November', 'dezembro': 'December'}

# extract the month name from the input string
month_name_pt = monthDay[2]

# convert the month name to English using the dictionary
month_name_en = month_names[month_name_pt]

# convert the month name to its corresponding number (e.g., 'March' -> 3)
month_number = datetime.strptime(month_name_en, '%B').month

# create the datetime object
todayDate = datetime(year=year, month=month_number, day=day)

In [ ]:
# Get SPX Options Data
df = pd.read_csv(filename, sep=",", header=None, skiprows=4)
df.columns = ['ExpirationDate','Calls','CallLastSale','CallNet','CallBid','CallAsk','CallVol',
              'CallIV','CallDelta','CallGamma','CallOpenInt','StrikePrice','Puts','PutLastSale',
              'PutNet','PutBid','PutAsk','PutVol','PutIV','PutDelta','PutGamma','PutOpenInt']


df['ExpirationDate'] = pd.to_datetime(df['ExpirationDate'], format='%a %b %d %Y')
df['ExpirationDate'] = df['ExpirationDate'] + timedelta(hours=16)
df['StrikePrice'] = df['StrikePrice'].astype(float)
df['CallIV'] = df['CallIV'].astype(float)
df['PutIV'] = df['PutIV'].astype(float)
df['CallGamma'] = df['CallGamma'].astype(float)
df['PutGamma'] = df['PutGamma'].astype(float)
df['CallOpenInt'] = df['CallOpenInt'].astype(float)
df['PutOpenInt'] = df['PutOpenInt'].astype(float)

In [ ]:
# ---=== CALCULATE SPOT GAMMA ===---
# Gamma Exposure = Unit Gamma * Open Interest * Contract Size * Spot Price
# To further convert into 'per 1% move' quantity, multiply by 1% of spotPrice
df['CallGEX'] = df['CallGamma'] * df['CallOpenInt'] * 100 * spotPrice * spotPrice * 0.01
df['PutGEX'] = df['PutGamma'] * df['PutOpenInt'] * 100 * spotPrice * spotPrice * 0.01 * -1

df['TotalGamma'] = (df.CallGEX + df.PutGEX) / 10**9
dfAgg = df.groupby(['StrikePrice']).sum(numeric_only=True)
strikes = dfAgg.index.values

In [ ]:
# Chart 1: Absolute Gamma Exposure
# define os dados
x_data = strikes
y_data = dfAgg['TotalGamma'].to_numpy()

# cria um gráfico de barras
fig = go.Figure(
    go.Bar(
        x=x_data,
        y=y_data,
        width=6,
        marker_color='rgb(26, 118, 255)',  # cor das barras
        marker_line_color='black',  # cor das linhas de contorno das barras
        marker_line_width=0.15,  # largura das linhas de contorno das barras
        name='Gamma Exposure'
    )
)

# adiciona uma linha vertical para o preço spot
fig.add_shape(
    type='line',
    x0=spotPrice,
    y0=min(y_data),
    x1=spotPrice,
    y1=max(y_data),
    line=dict(
        color='red',
        width=2,
        dash='dash'  # estilo da linha
    )
)

# define o layout do gráfico
fig.update_layout(
    title={
        'text': f"Total Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move",
        'font': {'size': 20, 'family': 'Arial Black',}
    },
    xaxis_title='Strike',
    yaxis_title='Spot Gamma Exposure ($ billions/1% move)',
    xaxis=dict(range=[fromStrike, toStrike]),
    yaxis=dict(tickformat='$,.2f'),
    plot_bgcolor='white',  # cor de fundo do gráfico
    font=dict(family='Arial', size=12, color='black')
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)


# mostra o gráfico
fig.show()

In [ ]:
# DADOS DO CHART 1
dfAgg_sorted = dfAgg.sort_values(by='TotalGamma')

# Get the 3 smallest gamma values
smallest_gamma = dfAgg_sorted.head(3)
print("Menores valores de Gamma:")
print(f"  Put Wall: {smallest_gamma.iloc[0]['TotalGamma']:.4f} at strike {smallest_gamma.index[0]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[1]['TotalGamma']:.4f} at strike {smallest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {smallest_gamma.iloc[2]['TotalGamma']:.4f} at strike {smallest_gamma.index[2]:.0f}")


# Get the 3 largest gamma values
largest_gamma = dfAgg_sorted.tail(3)
print("\nMaiores valores de Gamma:")
print(f"  Call Wall: {largest_gamma.iloc[2]['TotalGamma']:.4f} at strike {largest_gamma.index[2]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[1]['TotalGamma']:.4f} at strike {largest_gamma.index[1]:.0f}")
print(f"  Large Gamma: {largest_gamma.iloc[0]['TotalGamma']:.4f} at strike {largest_gamma.index[0]:.0f}")

# Find the positive gamma strike closest to the spot price (Vol Trigger)
positive_gamma_strikes = dfAgg[dfAgg['TotalGamma'] > 0].index
if len(positive_gamma_strikes) > 0:
    # Find the strike closest to the spot price among the positive gamma strikes
    vol_trigger_strike = positive_gamma_strikes[np.abs(positive_gamma_strikes - spotPrice).argmin()]
    vol_trigger_gamma = dfAgg.loc[vol_trigger_strike, 'TotalGamma']
    print(f"\nVol Trigger: {vol_trigger_gamma:.4f} at strike {vol_trigger_strike:.0f}")
else:
    print("\nNo Vol Trigger found (no positive gamma strikes).")


# Print the total gamma
print(f"\nTotal Gamma: ${df['TotalGamma'].sum():,.2f} Bn per 1% SPX Move")

Menores valores de Gamma:
  Put Wall: -2.1015 at strike 6500
  Large Gamma: -2.0658 at strike 6400
  Large Gamma: -1.7506 at strike 6300

Maiores valores de Gamma:
  Call Wall: 6.4540 at strike 7000
  Large Gamma: 5.2407 at strike 6800
  Large Gamma: 4.6087 at strike 6900

Vol Trigger: 0.0302 at strike 6700

Total Gamma: $10.41 Bn per 1% SPX Move


In [ ]:
# Chart 2: Absolute Gamma Exposure by Calls and Puts
fig = go.Figure()
fig.add_bar(x=strikes, y=dfAgg['CallGEX'].to_numpy() / 10**9, width=6, name="Call Gamma")
fig.add_bar(x=strikes, y=dfAgg['PutGEX'].to_numpy() / 10**9, width=6, name="Put Gamma")
fig.update_xaxes(range=[fromStrike, toStrike])
chartTitle = "Total Gamma: $" + str("{:.2f}".format(df['TotalGamma'].sum())) + " Bn per 1% SPX Move"
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))
fig.update_xaxes(title_text="Strike")
fig.update_yaxes(title_text="Spot Gamma Exposure ($ billions/1% move)")
fig.add_shape(dict(type="line", x0=spotPrice, y0=0, x1=spotPrice, y1=max(dfAgg['CallGEX'].to_numpy() / 10**9), line=dict(color="black", width=2), name="SPX Spot:" + str("{:,.0f}".format(spotPrice))))

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1750,
    height=800
)

fig.show()


In [ ]:
# DADOS CHART 2
dfAgg['AbsoluteTotalGEX'] = dfAgg['CallGEX'].abs() + dfAgg['PutGEX'].abs()

# Sort by AbsoluteTotalGEX to find the strikes with the largest combined exposure
dfAgg_sorted_gex = dfAgg.sort_values(by='AbsoluteTotalGEX', ascending=False)

# Get the top 6 strikes based on combined absolute GEX
gex_levels = dfAgg_sorted_gex.head(6)

print("Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):")
for i in range(len(gex_levels)):
    strike = gex_levels.index[i]
    call_gex = gex_levels.iloc[i]['CallGEX'] / 10**9
    put_gex = gex_levels.iloc[i]['PutGEX'] / 10**9
    print(f"  GEX Level {i+1}: Strike {strike:.0f}, Call GEX: {call_gex:.4f} Bn, Put GEX: {put_gex:.4f} Bn")

Top 6 GEX Levels (based on sum of absolute Call and Put Gamma Exposure):
  GEX Level 1: Strike 6000, Call GEX: 10.4459 Bn, Put GEX: -11.9455 Bn
  GEX Level 2: Strike 6700, Call GEX: 10.0212 Bn, Put GEX: -9.9910 Bn
  GEX Level 3: Strike 7000, Call GEX: 9.3668 Bn, Put GEX: -2.9128 Bn
  GEX Level 4: Strike 6750, Call GEX: 7.5394 Bn, Put GEX: -4.4004 Bn
  GEX Level 5: Strike 6600, Call GEX: 5.2149 Bn, Put GEX: -6.2754 Bn
  GEX Level 6: Strike 6650, Call GEX: 5.5182 Bn, Put GEX: -5.9513 Bn


In [ ]:
# ---=== CALCULATE GAMMA PROFILE ===---
levels = np.linspace(fromStrike, toStrike, 60)

# For 0DTE options, I'm setting DTE = 1 day, otherwise they get excluded
df['daysTillExp'] = [1/262 if (np.busday_count(todayDate.date(), x.date())) == 0 \
                           else np.busday_count(todayDate.date(), x.date())/262 for x in df.ExpirationDate]

nextExpiry = df['ExpirationDate'].min()

df['IsThirdFriday'] = [isThirdFriday(x) for x in df.ExpirationDate]
thirdFridays = df.loc[df['IsThirdFriday'] == True]
nextMonthlyExp = thirdFridays['ExpirationDate'].min()

totalGamma = []
totalGammaExNext = []
totalGammaExFri = []

In [ ]:
# For each spot level, calc gamma exposure at that point
for level in levels:
    df['callGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['CallIV'],
                                                          row['daysTillExp'], 0, 0, "call", row['CallOpenInt']), axis = 1)

    df['putGammaEx'] = df.apply(lambda row : calcGammaEx(level, row['StrikePrice'], row['PutIV'],
                                                         row['daysTillExp'], 0, 0, "put", row['PutOpenInt']), axis = 1)

    totalGamma.append(df['callGammaEx'].sum() - df['putGammaEx'].sum())

    exNxt = df.loc[df['ExpirationDate'] != nextExpiry]
    totalGammaExNext.append(exNxt['callGammaEx'].sum() - exNxt['putGammaEx'].sum())

    exFri = df.loc[df['ExpirationDate'] != nextMonthlyExp]
    totalGammaExFri.append(exFri['callGammaEx'].sum() - exFri['putGammaEx'].sum())

totalGamma = np.array(totalGamma) / 10**9
totalGammaExNext = np.array(totalGammaExNext) / 10**9
totalGammaExFri = np.array(totalGammaExFri) / 10**9

In [ ]:
# Find Gamma Flip Point
zeroCrossIdx = np.where(np.diff(np.sign(totalGamma)))[0]

negGamma = totalGamma[zeroCrossIdx]
posGamma = totalGamma[zeroCrossIdx+1]
negStrike = levels[zeroCrossIdx]
posStrike = levels[zeroCrossIdx+1]

zeroGamma = posStrike - ((posStrike - negStrike) * posGamma/(posGamma-negGamma))
zeroGamma = zeroGamma[0]

In [ ]:
# Chart 3: Gamma Exposure Profile
fig = go.Figure()

fig.add_trace(go.Scatter(x=levels, y=totalGamma, mode='lines', name='All Expiries'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExNext, mode='lines', name='Ex-Next Expiry'))
fig.add_trace(go.Scatter(x=levels, y=totalGammaExFri, mode='lines', name='Ex-Next Monthly Expiry'))

chartTitle = "Gamma Exposure Profile, SPX, " + todayDate.strftime('%d %b %Y')
fig.update_layout(title=chartTitle, xaxis_title='Index Price', yaxis_title='Gamma Exposure ($ billions/1% move)')
fig.update_layout(title_text=chartTitle, title_font=dict(size=20, family="Arial Black"))

fig.add_shape(
    dict(
        type="line",
        x0=spotPrice,
        y0=min(totalGamma),
        x1=spotPrice,
        y1=max(totalGamma),
        line=dict(color="red", width=1.5),
        name="SPX Spot: " + str("{:,.0f}".format(spotPrice))
    )
)

fig.add_shape(
    dict(
        type="line",
        x0=zeroGamma,
        y0=min(totalGamma),
        x1=zeroGamma,
        y1=max(totalGamma),
        line=dict(color="green", width=1.5),
        name="Gamma Flip: " + str("{:,.0f}".format(zeroGamma))
    )
)

fig.update_xaxes(range=[fromStrike, toStrike])
fig.update_yaxes(range=[min(totalGamma), max(totalGamma)])

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[min(totalGamma), min(totalGamma), min(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="red",
        opacity=0.1,
        showlegend=False,
        name="Negative Gamma"
    )
)

fig.add_trace(
    go.Scatter(
        x=[fromStrike, zeroGamma, toStrike],
        y=[max(totalGamma), max(totalGamma), max(totalGamma)],
        mode="none",
        fill="toself",
        fillcolor="green",
        opacity=0.1,
        showlegend=False,
        name="Positive Gamma"
    )
)

# DEFINIR O TAMANHO DO GRÁFICO EM PIXELS
fig.update_layout(
    width=1400,
    height=700
)

fig.show()

In [ ]:
# IDENTIFICAR E ROTULAR PONTOS NA LINHA 'Ex-Next Monthly Expiry' FORA DO CHART 3
# Gamma Flip (primeiro valor acima da linha verde central)
gamma_flip_index = np.where(totalGammaExFri > 0)[0][0] if np.any(totalGammaExFri > 0) else None
if gamma_flip_index is not None:
    gamma_flip_strike = levels[gamma_flip_index]
    gamma_flip_value = totalGammaExFri[gamma_flip_index]
    print(f"Gamma Flip (Ex-Next Monthly Expiry): {gamma_flip_value:.4f} at strike {gamma_flip_strike:.0f}")
else:
    print("No Gamma Flip found (Ex-Next Monthly Expiry).")


# Max Gamma Positivo
max_gamma_positive_value = np.max(totalGammaExFri)
max_gamma_positive_index = np.argmax(totalGammaExFri)
max_gamma_positive_strike = levels[max_gamma_positive_index]
print(f"Max Gamma Positivo (Ex-Next Monthly Expiry): {max_gamma_positive_value:.4f} at strike {max_gamma_positive_strike:.0f}")


# Min Gamma Negativo
min_gamma_negative_value = np.min(totalGammaExFri)
min_gamma_negative_index = np.argmin(totalGammaExFri)
min_gamma_negative_strike = levels[min_gamma_negative_index]
print(f"Min Gamma Negativo (Ex-Next Monthly Expiry): {min_gamma_negative_value:.4f} at strike {min_gamma_negative_strike:.0f}")

Gamma Flip (Ex-Next Monthly Expiry): 10.0663 at strike 6722
Max Gamma Positivo (Ex-Next Monthly Expiry): 64.1463 at strike 6904
Min Gamma Negativo (Ex-Next Monthly Expiry): -65.4512 at strike 5814
